# Caribu for developers, 2: extend the component

**What you will learn:** add a state variable and a step to `CaribuLight` by subclassing it, make the environment
model light with it, build a scene by hand (a plant model, a planting table, a `Scene`), and test the new variable.

**You need:** developer tutorial 1 (the component formalism). **Run time:** a few seconds.

## 1. A new variable and its step

`CaribuLight` gives each element the PAR it absorbs **per unit area** (`PARa`, µmol m⁻² s⁻¹). A photosynthesis model
working per element may need the PAR the element absorbs **in all** (µmol s⁻¹): `PARa` times its lit area. A
subclass declares it as a state variable and computes it in a step:

- `state_variable` declares it: its unit, its description, how it is aggregated (`extensive`: summed over coarser
  scales), its initial value and its scale (`SubOrgan`, the plant's elements);
- the method `_PAR_flux` computes it. Its arguments name the variables it reads, and the dictionary it returns, the
  variables it writes;
- `@state(include_inactive=True)` places it in the row "state", after `_absorbed_PAR`. `CaribuLightModel` runs
  that row at every step, also between two lightings, so the new variable follows the PAR above the canopy;
- `@dataclass` is needed on the subclass, as on every component.

In [1]:
from dataclasses import dataclass

import numpy as np

from openalea.caribu.component import CaribuLight, CaribuLightModel
from openalea.metafspm.coupling.component import state_variable
from openalea.metafspm.data_structure.configs import ScalesConfig
from openalea.metafspm.solve.decorator import state


@dataclass
class CaribuLightWithFlux(CaribuLight):
    PAR_flux: float = state_variable(unit="µmol s-1", description="The PAR the element absorbs, over its whole area",
                                     kind="extensive", default=0., scale=ScalesConfig.SubOrgan)

    @state(include_inactive=True)
    def _PAR_flux(self, PARa, lit_area) -> dict:
        return {"PAR_flux": PARa * lit_area}


class LightWithFlux(CaribuLightModel):
    """The environment model lighting with the extended component."""
    light_component = CaribuLightWithFlux

## 2. Where it appears

The new variable is in the declarations, and the new step in the schedule:

In [2]:
declarations = CaribuLightWithFlux.declarations()
print(declarations[declarations["name"] == "PAR_flux"][["name", "kind", "scale", "unit", "description"]].to_string())
print()
for stages, steps in CaribuLightWithFlux.schedule():
    print(f"{', '.join(stages):>6}  {', '.join(steps)}")

        name       kind     scale      unit                                       description
33  PAR_flux  extensive  SubOrgan  µmol s-1  The PAR the element absorbs, over its whole area

  rate  _optics, _relative_light
 state  _absorbed_PAR, _PAR_flux


## 3. A scene by hand

`simulate` (the user tutorials) builds its scene itself. Here the three layers are written out:

- the **plant model** is `SimplePlantModel` (in `openalea.caribu.simulation`), a metafspm `CompositeModel`. Its
  initiator, `SimplePlant`, builds each plant's stem and leaves; `PlantOptics` gives them their optics, which the
  light reads, and `AbsorbedLight` adds up the PAR each one absorbs;
- the **planting table** gives each plant its position and its scenario (here the default parameters), and the
  stand's size: the infinite canopy's pattern;
- the **scene** holds the plants and the environment model `LightWithFlux`, its calendar (`start=(day of year,
  hour)`: the sun's place at its first step) and the forcing `PARi`, the PAR above the canopy at each time (in
  seconds since the start). The light's variables are the light model's, on the union of the plants' elements.

In [3]:
from openalea.caribu.simulation import SimplePlantModel
from openalea.metafspm.scene.population import Planting, planting
from openalea.metafspm.scene.scene import Scene

plants = [Planting(x=0.05 * (column + 0.5), y=0.15 * (row + 0.5), rotation=37. * (4 * row + column))
          for row in range(2) for column in range(4)]
table = planting(plants, SimplePlantModel, {}, xrange=0.2, yrange=0.3)
scene = Scene(table, environment=[LightWithFlux], time_step=3600., forcings={"PARi": lambda time: 1000.},
              start=(172., 11.))
scene.run()
ds = scene.environment[0].data_structure                  # the light model's: the elements' union
flux = np.asarray(ds.get("PAR_flux"))
print(f"{ds.n_nodes()} elements absorb {flux.sum():.1f} µmol s-1 in all, under 1000 µmol m-2 s-1 above "
      f"{0.2 * 0.3:.2f} m² of ground")

40 elements absorb 40.1 µmol s-1 in all, under 1000 µmol m-2 s-1 above 0.06 m² of ground


## 4. Test it

A test states what the step must give, on an independent computation. In the package it would go in `test/`, run by
pytest; here it runs in place:

In [4]:
def test_flux_is_the_absorbed_PAR_over_the_area():
    PARa, area = np.asarray(ds.get("PARa")), np.asarray(ds.get("lit_area"))
    np.testing.assert_allclose(flux, PARa * area, rtol=1e-12)
    assert flux.sum() <= 1000. * 0.2 * 0.3                       # no more than what falls on the ground


test_flux_is_the_absorbed_PAR_over_the_area()
print("passed")

passed


## 5. Further

- A variable another model writes is declared with `input_variable(..., by=...)`. `CaribuLight`'s
  `PAR_reflectance` and `PAR_transmittance` are such inputs: the plant model's optics (`PlantOptics`, or any
  optical model) give them, linked by name.
- The light's outputs reach a plant model by name too: a component of it declaring an input `PARa` reads the light's
  (the scene's default identity links; `scene.translator.links` lists them). Several plant populations (an intercrop)
  share the light in the same union, each linked to its part (`test/test_component.py`).
- A new light model, rather than a new variable, belongs in the modules under the component (`direct.py`,
  `nested.py`, ...), with its equations in `doc/equations.rst` and its tests against an independent computation.